# 5. Treino e experimentos

← [Modelos](04_models.ipynb) · [índice](README.pt-BR.md) · [Curvas de treinamento](06_training_curves.ipynb) →

Laço de treino, definição dos 17 experimentos e tabela de resultados.

## Laço de treino

- **Função de perda:** cross-entropy, isto é, softmax seguida do negativo do log da probabilidade da classe
  correta (classificador softmax). Os gradientes saem por backpropagation, com a
  regra da cadeia aplicada ao grafo computacional, via `loss.backward()` (autograd do PyTorch).
- **Mini-batch gradient descent** com batch de 128, um valor comum na prática e
  igual ao batch da AlexNet. Os experimentos E14 e E15 testam 32 e 512.
- **Otimizadores:** SGD, SGD com momentum (ρ = 0,9) e **Adam**
  (β₁ = 0,9, β₂ = 0,999 e lr = 1e-3, o ponto de partida mais usado).
- **Decaimento da taxa de aprendizado:**
  - `plateau` divide a lr por 10 quando a loss de validação para de melhorar. É inspirado na AlexNet, que
    reduz a lr por 10 quando a acurácia de validação para de aumentar; aqui o critério é a loss;
  - `onecycle` faz um aquecimento curto e depois um decaimento em cosseno até quase zero. Outras formas comuns são
    decaimento em degraus, exponencial e 1/t, mas a ideia é a mesma: lr alta no começo para avançar rápido e
    lr baixa no fim para convergir.
- **SGD com lr = 0,05:** é um valor maior que o 1e-2 da AlexNet, o que dá certo porque o BatchNorm permite
  taxas de aprendizado mais altas.
- **Sem regularização L2** (`weight_decay = 0`): a regularização fica por conta do Dropout, da data
  augmentation e do early stopping. Assim, os otimizadores são comparados nas mesmas condições.
- **Early stopping:** guardo os pesos da época com a maior acurácia de validação e
  paro o treino se ela não melhorar por `patience` épocas.
- As métricas de treino de cada época são recalculadas em modo `eval`, sem dropout e sem augmentation, para
  que treino e validação sejam comparados nas mesmas condições.

In [1]:
import inspect
from IPython.display import Code, display

def show(*objs):
    """Mostra o código-fonte das funções/classes (definidas em svhn_utils.py)."""
    display(Code("\n\n".join(inspect.getsource(o) for o in objs), language="python"))

import numpy as np
import pandas as pd

from svhn_utils import (EXPERIMENTS, RESULTS_DIR, get_device, load_data, make_optimizer, make_scheduler,
                        run_experiment, train)

show(make_optimizer, make_scheduler, train)

def make_optimizer(name, params, lr, weight_decay=0.0):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, alpha=0.99, weight_decay=weight_decay)
    raise ValueError(name)


def make_scheduler(name, opt, epochs, steps_per_epoch):
    if name == "none":
        return None, None
    if name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.1, patience=2), "epoch"
    if name == "onecycle":
        return torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=opt.param_groups[0]["lr"], total_steps=epochs * steps_per_epoch, pct_start=0.15), "step"
    raise ValueError(name)


def train(model, data, cfg, device, verbose=True):
    set_seed(cfg["seed"])
    Xtr, ytr = (t.to(device) for t in data["train"])
    Xva, yva = (t.to(device) for t in data["val"])
    norm = Normalizer(*channel_stats(data["train"][0]), device)

    model = model.to(device)
    opt = make_optimizer(cfg["optimizer"], model.parameters(), cfg["lr"], cfg["weight_decay"])
    steps = math.ceil(len(ytr) / cfg["batch_size"])
    sched, sched_mode = make_scheduler(cfg["scheduler"], opt, cfg["epochs"], steps)

    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "time"]}
    best_acc, best_state, best_epoch, bad = -1.0, None, 0, 0
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t0 = time.time()
        for xb, yb in iterate(Xtr, ytr, cfg["batch_size"], shuffle=True):
            x = (augment(xb.float() / 255.0) - norm.mean) / norm.std if cfg["augment"] else norm(xb)
            loss = F.cross_entropy(model(x), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            if sched_mode == "step":
                sched.step()

        tr_loss, tr_acc = evaluate(model, Xtr, ytr, norm)
        va_loss, va_acc = evaluate(model, Xva, yva, norm)
        hist["lr"].append(opt.param_groups[0]["lr"])
        if sched_mode == "epoch":
            sched.step(va_loss)
        for k, v in zip(["train_loss", "train_acc", "val_loss", "val_acc", "time"],
                        [tr_loss, tr_acc, va_loss, va_acc, time.time() - t0]):
            hist[k].append(v)
        if verbose:
            print(f"  ep {epoch:3d} | treino loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {hist['lr'][-1]:.1e} | {hist['time'][-1]:.1f}s")

        if va_acc > best_acc:
            best_acc, best_epoch, bad = va_acc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= cfg["patience"]:
                if verbose:
                    print(f"  early stopping na época {epoch} (melhor época: {best_epoch})")
                break

    model.load_state_dict(best_state)
    hist["best_epoch"], hist["epochs_run"] = best_epoch, len(hist["train_loss"])
    return model, norm, hist

## Experimentos

Em geral, cada experimento muda uma única coisa em relação ao experimento da coluna "base". Assim, a
diferença de acurácia mostra o efeito de cada modificação isoladamente. A configuração padrão é Adam com
lr = 1e-3, batch de 128, até 30 épocas, *early stopping* com paciência de 8 épocas e semente 0.

| Grupo | Experimentos | O que avalia |
|---|---|---|
| MLP | E1 e E2 | baseline totalmente conectada e versão com BatchNorm e Dropout |
| CNN | E3 a E7 | CNN baseline, depois a VGG, e em seguida BatchNorm, Dropout e data augmentation, um de cada vez |
| Otimização | E8 a E15 | SGD com momentum, SGD puro, decaimento da lr, OneCycle, lr alta e baixa, batch 32 e 512 |
| Capacidade | E16 e E17 | mais filtros e menos camadas |

In [2]:
pd.DataFrame([{"Exp.": e["id"], "Modificação": e["desc"], "Base": e["base"] or "-",
               **{k: e["train"][k] for k in ("optimizer", "lr", "batch_size", "scheduler", "augment", "epochs", "patience")}}
              for e in EXPERIMENTS])

,Exp.,Modificação,Base,optimizer,lr,batch_size,scheduler,augment,epochs,patience
0,E1,MLP baseline (3072-512-256-10),-,adam,0.0010,128,none,False,30,8
1,E2,MLP + BatchNorm + Dropout 0.3,E1,adam,0.0010,128,none,False,30,8
2,E3,CNN baseline (LeNet-like),E1,adam,0.0010,128,none,False,30,8
3,E4,CNN VGG-like (3 blocos × 2 conv3x3),E3,adam,0.0010,128,none,False,30,8
4,E5,+ BatchNorm,E4,adam,0.0010,128,none,False,30,8
5,E6,"+ Dropout (0.5 FC, 0.1 conv)",E5,adam,0.0010,128,none,False,30,8
6,E7,+ Data Augmentation,E6,adam,0.0010,128,none,True,40,10
7,E8,SGD+Momentum 0.9 (lr 0.05),E7,sgd_momentum,0.0500,128,none,True,40,10
8,E9,SGD sem momentum (lr 0.05),E8,sgd,0.0500,128,none,True,40,10
9,E10,+ LR decay (÷10 no platô),E7,adam,0.0010,128,plateau,True,40,10


## Execução

Com `REFAZER_EXPERIMENTOS = False`, os resultados salvos em `outputs/results/` são carregados e nada é
retreinado. Com `True`, os 17 experimentos são treinados de novo (cerca de 3 horas numa GPU Apple M5).

In [3]:
REFAZER_EXPERIMENTOS = False   # True = treina novamente mesmo com resultado salvo

device = get_device()
data, _ = load_data()
results = {e["id"]: run_experiment(e, data, device, redo=REFAZER_EXPERIMENTOS) for e in EXPERIMENTS}

E1: carregado do disco | val acc 0.8258 | teste acc 0.8002
E2: carregado do disco | val acc 0.8567 | teste acc 0.8385
E3: carregado do disco | val acc 0.8918 | teste acc 0.8842
E4: carregado do disco | val acc 0.9253 | teste acc 0.9270
E5: carregado do disco | val acc 0.9302 | teste acc 0.9262
E6: carregado do disco | val acc 0.9503 | teste acc 0.9529
E7: carregado do disco | val acc 0.9574 | teste acc 0.9645
E8: carregado do disco | val acc 0.9554 | teste acc 0.9658
E9: carregado do disco | val acc 0.9472 | teste acc 0.9597
E10: carregado do disco | val acc 0.9565 | teste acc 0.9686
E11: carregado do disco | val acc 0.9575 | teste acc 0.9684
E12: carregado do disco | val acc 0.9543 | teste acc 0.9637
E13: carregado do disco | val acc 0.9453 | teste acc 0.9576
E14: carregado do disco | val acc 0.9545 | teste acc 0.9653
E15: carregado do disco | val acc 0.9535 | teste acc 0.9639
E16: carregado do disco | val acc 0.9607 | teste acc 0.9713
E17: carregado do disco | val acc 0.9420 | teste 

## Tabela de resultados

In [4]:
rows = []
for e in EXPERIMENTS:
    r = results[e["id"]]
    base = results.get(e["base"]) if e["base"] else None
    delta = (r["val_acc"] - base["val_acc"]) * 100 if base else None
    rows.append({
        "Exp.": r["id"], "Modificação": r["desc"], "Base": e["base"] or "-",
        "Parâmetros": f"{r['params']:,}", "Melhor época": f"{r['best_epoch']}/{r['epochs_run']}",
        "Loss treino": round(r["train_loss"], 4), "Loss val.": round(r["val_loss"], 4),
        "Acc treino (%)": round(r["train_acc"] * 100, 2), "Acc val. (%)": round(r["val_acc"] * 100, 2),
        "Acc teste (%)": round(r["test_acc"] * 100, 2),
        "Δ val vs base (p.p.)": "-" if delta is None else f"{delta:+.2f}",
        "s/época": round(r["sec_per_epoch"], 1),
    })
table = pd.DataFrame(rows)
table.to_csv(RESULTS_DIR / "results_table.csv", index=False)
(RESULTS_DIR / "results_table.md").write_text(table.to_markdown(index=False))
table

,Exp.,Modificação,Base,Parâmetros,Melhor época,Loss treino,Loss val.,Acc treino (%),Acc val. (%),Acc teste (%),Δ val vs base (p.p.),s/época
0,E1,MLP baseline (3072-512-256-10),-,"1,707,274",25/30,0.2521,0.7775,91.94,82.58,80.02,-,1.2
1,E2,MLP + BatchNorm + Dropout 0.3,E1,"1,708,042",29/30,0.3053,0.4763,91.16,85.67,83.85,+3.08,1.1
2,E3,CNN baseline (LeNet-like),E1,"1,105,098",9/17,0.0847,0.4819,97.51,89.18,88.42,+6.59,30.7
3,E4,CNN VGG-like (3 blocos × 2 conv3x3),E3,"814,122",6/14,0.0867,0.3032,97.45,92.53,92.70,+3.36,14.8
4,E5,+ BatchNorm,E4,"814,826",19/27,0.0127,0.3726,99.58,93.02,92.62,+0.49,20.0
5,E6,"+ Dropout (0.5 FC, 0.1 conv)",E5,"814,826",27/30,0.0372,0.2008,99.10,95.03,95.29,+2.01,20.9
6,E7,+ Data Augmentation,E6,"814,826",33/40,0.0975,0.1667,97.45,95.74,96.45,+0.71,16.7
7,E8,SGD+Momentum 0.9 (lr 0.05),E7,"814,826",33/40,0.1079,0.1667,97.21,95.54,96.58,-0.20,17.7
8,E9,SGD sem momentum (lr 0.05),E8,"814,826",39/40,0.1513,0.1878,95.88,94.72,95.97,-0.82,17.1
9,E10,+ LR decay (÷10 no platô),E7,"814,826",36/40,0.0887,0.1614,97.77,95.65,96.86,-0.10,16.7
